In [13]:
import numpy as np
from sentence_transformers import SentenceTransformer


class VectorDatabase:
    def __init__(self, model_name = "all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)
        self.documents = []
        self.embeddings = None

    
    def upsert(self, texts: list[str]):
        new_embeddings = self.model.encode(texts)
        self.documents.extend(texts)

        if  self.embeddings is None:
            self.embeddings = new_embeddings
        else:
            self.embeddings = np.vstack([self.embeddings, new_embeddings])
        
        return f"{len(self.embeddings)} Added to vector database successfully"
    

    def search(self, query:str, top_k = 5):
        q_embedding = self.model.encode([query])[0]
        dot_products = np.dot(self.embeddings, q_embedding)
        norms = np.linalg.norm(self.embeddings, axis=1) * np.linalg.norm(q_embedding)

        similarities = dot_products / norms

        print(f"documents of {len(self.documents)} vs similarity: {len(similarities)}")

        top_indices = np.argsort(similarities)[::-1][:top_k]
        return [(self.documents[i], float(similarities[i])) for i in top_indices]

In [14]:
db = VectorDatabase()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4561.58it/s]


In [15]:
def text_splitter(text: str, chunk_size: int = 150, overlap: int = 30) -> list[str]:
    words = text.split()
    chunks = []
    current_chunk = []
    current_length = 0

    for word in words:
        # +1 accounts for the space between words
        if current_length + len(word) + 1 > chunk_size and current_chunk:
            chunks.append(" ".join(current_chunk))
            
            # Retain words from the end of the current chunk for overlap
            overlap_chunk = []
            overlap_len = 0
            for w in reversed(current_chunk):
                if overlap_len + len(w) + 1 <= overlap:
                    overlap_chunk.insert(0, w)
                    overlap_len += len(w) + 1
                else:
                    break
            
            current_chunk = overlap_chunk
            current_length = overlap_len

        current_chunk.append(word)
        current_length += len(word) + 1

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks


In [16]:
with open("test_docs.txt") as f:
    document = f.read()

In [17]:
chunks = text_splitter(document, chunk_size=200,overlap=50)

In [18]:
chunks[:5]

['ABSTRACT This project proposes the design and implementation of a web-based Blood Donor Registration and Information Management System for Muslim Hospital, Saki. The system will provide a structured',
 'Saki. The system will provide a structured platform for registering voluntary blood donors, maintaining donor profiles, recording verified blood group information and donation history, and retrieving',
 'information and donation history, and retrieving donor records when required. The study is motivated by information-management challenges that can arise when records are held in paper files or',
 'can arise when records are held in paper files or disconnected systems, including slow retrieval, duplicate entries, outdated contact details, and difficulty preparing summaries. The hospitalâ€™s',
 'difficulty preparing summaries. The hospitalâ€™s actual workflow and challenges will first be established through approved requirements gathering. The project will adopt an iterative']

In [19]:
db.upsert(chunks)

'95 Added to vector database successfully'

In [22]:
for i in db.search("What is Blood?"):
    print("--------------------")
    print(i[0])

documents of 95 vs similarity: 95
--------------------
usable, secure, and maintainable application for organising and retrieving donor information. CHAPTER ONE INTRODUCTION 1.1 Background to the Study Blood is an essential healthcare resource and may be
--------------------
Management Blood donor information management involves collecting, organising, updating, retrieving, and protecting information about people who have expressed interest in donating blood or have
--------------------
information; recording blood group information as supplied or verified under hospital procedures; secure staff login; role-based administration; viewing and updating donor profiles; authorised
--------------------
1.3 Aim and Objectives of the Study The aim is to design and implement a web-based Blood Donor Registration and Information Management System for Muslim Hospital, Saki. The specific objectives are
--------------------
ABSTRACT This project proposes the design and implementation of a web-based 